# 8.5 Binary Files

**Prerequisites:** 8.1 File Handling — Text, 2.1 Strings (str vs bytes), 4.3 Generators  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Text mode vs binary mode — what the text layer actually does
- Chunked reading, so file size stops mattering
- Hashing a file with `hashlib` — checksums and deduplication
- Copying efficiently with `shutil.copyfileobj`
- **Magic bytes** — identifying a file from its first few bytes
- `struct` for fixed-width binary records
- `BytesIO` for in-memory binary

---

## Text mode vs binary mode

`open()` has two families of modes, and the difference is not cosmetic.

| | Text mode (`"r"`, `"w"`) | Binary mode (`"rb"`, `"wb"`) |
|---|---|---|
| You get / give | `str` | `bytes` |
| Encoding applied | **Yes** — `encoding=` | **No** |
| Newline translation | **Yes** — `\r\n` ↔ `\n` | **No** |
| Safe for images, PDFs, archives | ❌ | ✅ |

Text mode adds a **translation layer**. It decodes bytes into `str` on the way in, encodes on
the way out, and rewrites line endings. That is exactly what you want for text and exactly
what corrupts everything else.

Binary mode removes the layer. What is on disk is what you get.

### When to use which

- **Text mode** — `.txt`, `.csv`, `.json`, `.py`, source code, logs
- **Binary mode** — images, PDFs, `.zip`, `.pkl`, executables, network protocol data, *and
  any file whose format you do not control*

In [ ]:
from pathlib import Path

demo = Path("File2Save/mode_demo.bin")

# Bytes that are NOT valid UTF-8, and that contain CR and LF
payload = bytes([0x00, 0xFF, 0x0D, 0x0A, 0x41, 0x80, 0x42])
demo.write_bytes(payload)

print("on disk        :", payload, f"({len(payload)} bytes)")

# ---- Binary mode: exactly what is on disk ----
with open(demo, "rb") as handle:
    raw = handle.read()
print("read as binary :", raw, f"({len(raw)} bytes)  identical: {raw == payload}")

# ---- Text mode: fails outright, because these bytes are not valid UTF-8 ----
try:
    with open(demo, "r", encoding="utf-8") as handle:
        handle.read()
except UnicodeDecodeError as exc:
    print("read as text   : UnicodeDecodeError at byte", exc.start)

# ---- Text mode with a lenient encoding: 'succeeds', but silently changes the data ----
with open(demo, "r", encoding="latin-1", newline=None) as handle:
    as_text = handle.read()
print("as latin-1     :", repr(as_text))
print("  length is now", len(as_text), "characters, and \\r\\n became", repr(as_text[2:3]))
print("  ^ the CRLF was translated - the data is no longer what was on disk")

demo.unlink()

print("""
This is why you never open an image, a PDF or a zip in text mode.
It may not raise. It will corrupt.
""")

### Chunked reading: making file size irrelevant

`file.read()` loads the entire file into memory. For a 4 GB video that is a `MemoryError`.

The fix is to read in fixed-size **chunks**. The walrus operator (**1.4**) makes the loop
read cleanly:

```python
with open(path, "rb") as handle:
    while chunk := handle.read(65536):
        process(chunk)
```

A **generator** (**4.3**) turns that into a reusable building block — and once you have one,
hashing, copying and scanning are all the same three lines.

> **Chunk size:** 64 KB (`65536`) is a good default. Too small and you pay syscall overhead;
> too large and you lose the memory benefit.

In [ ]:
from pathlib import Path

source = Path("File2Save/Image.jpg")
print("file size:", f"{source.stat().st_size:,} bytes")


def chunks(path: Path, size: int = 65536):
    """Yield the file's contents in fixed-size byte chunks."""
    with open(path, "rb") as handle:
        while chunk := handle.read(size):
            yield chunk


# ---- Constant memory, whatever the file size ----
total = 0
count = 0
for chunk in chunks(source, size=8192):
    total += len(chunk)
    count += 1

print(f"read {total:,} bytes in {count} chunks of at most 8192")
print("matches file size:", total == source.stat().st_size)

# ---- Because it is a generator, it composes ----
from itertools import islice

first_two = list(islice(chunks(source, 16), 2))
print("\nfirst two 16-byte chunks:")
for c in first_two:
    print("  ", c.hex(" "))

# ---- Peak memory: whole-file vs chunked ----
import sys
whole = source.read_bytes()
print(f"\nwhole file in memory : {sys.getsizeof(whole):,} bytes")
print(f"one 64 KB chunk      : {sys.getsizeof(b'x' * 65536):,} bytes")
print("  ^ and that stays flat for a 4 GB file")

### Hashing a file

A **hash** is a fixed-length fingerprint of a file's contents. Two files with the same hash
are, for practical purposes, identical; one changed byte changes the hash completely.

| Use | Algorithm |
|---|---|
| Verifying a download matches what was published | **SHA-256** |
| Deduplicating files | SHA-256, or BLAKE2 (faster) |
| Detecting accidental corruption | SHA-256, or CRC32 for speed |
| ⚠️ Passwords | **None of these** — use `scrypt`/`argon2`/`bcrypt` |

> **MD5 and SHA-1 are broken** for security purposes — collisions can be constructed
> deliberately. They are still fine for "did this file change by accident", but never for
> anything an attacker could influence.

`hashlib` objects are fed incrementally with `.update()`, which is why chunked reading and
hashing fit together perfectly — you can hash a file of any size in constant memory.

In [ ]:
import hashlib
from pathlib import Path

source = Path("File2Save/Image.jpg")


def file_hash(path: Path, algorithm: str = "sha256", chunk_size: int = 65536) -> str:
    """Hash a file of any size in constant memory."""
    digest = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


for algo in ["md5", "sha1", "sha256", "blake2b"]:
    print(f"  {algo:<9} {file_hash(source, algo)[:48]}...")

# ---- Python 3.11+ has a built-in helper that does exactly this ----
with open(source, "rb") as handle:
    builtin = hashlib.file_digest(handle, "sha256").hexdigest()
print("\nhashlib.file_digest matches ours:", builtin == file_hash(source))


# ---- Verifying a copy ----
copy = Path("File2Save/Image_copy.jpg")
copy.write_bytes(source.read_bytes())
print("\ncopy verified :", file_hash(source) == file_hash(copy))

# ---- One changed byte changes everything ----
tampered = Path("File2Save/tampered.jpg")
data = bytearray(source.read_bytes())
data[1000] ^= 0x01                      # flip a single bit
tampered.write_bytes(data)

print("original :", file_hash(source)[:32])
print("tampered :", file_hash(tampered)[:32])
print("same size:", source.stat().st_size == tampered.stat().st_size)
print("  ^ identical size, one bit different, completely different hash")

tampered.unlink()


# ---- Deduplication: find identical files by content, not by name ----
folder = Path("File2Save")
seen: dict[str, list[str]] = {}
for path in sorted(folder.iterdir()):
    if path.is_file():
        seen.setdefault(file_hash(path), []).append(path.name)

print("\nduplicate groups in File2Save:")
found = False
for digest, names in seen.items():
    if len(names) > 1:
        print(f"  {digest[:16]}... -> {names}")
        found = True
if not found:
    print("  (none)")

### Copying efficiently, and `BytesIO`

`shutil.copyfileobj(src, dst)` copies between any two file-like objects in chunks. It works
on real files, network responses (**8.4**), and in-memory buffers — anything with `.read()`
and `.write()` (**5.4**, duck typing).

**`io.BytesIO`** is an in-memory binary file. Use it when a library demands a file object but
you have — or want — bytes in memory: building a file to upload, capturing binary output,
or testing without touching disk.

In [ ]:
import io, shutil
from pathlib import Path

source = Path("File2Save/Image.jpg")
target = Path("File2Save/Image_copyfileobj.jpg")

# ---- copyfileobj: chunked copy between any two file-like objects ----
with open(source, "rb") as src, open(target, "wb") as dst:
    shutil.copyfileobj(src, dst, length=65536)

print("copied:", target.stat().st_size, "bytes | identical:",
      source.read_bytes() == target.read_bytes())
target.unlink()

# shutil.copy() is the one-liner when both ends are paths
shutil.copy(source, "File2Save/Image_copy.jpg")
print("shutil.copy also works for the simple case")


# ---- BytesIO: a file that lives in memory ----
buffer = io.BytesIO()
buffer.write(b"header:")
buffer.write(b"\x00\x01\x02")
print("\nbuffer contents:", buffer.getvalue())
print("position       :", buffer.tell())

buffer.seek(0)                          # rewind, exactly like a real file
print("read from start:", buffer.read(7))

# It satisfies the same protocol, so copyfileobj accepts it
memory_copy = io.BytesIO()
with open(source, "rb") as src:
    shutil.copyfileobj(src, memory_copy)
print("\ncopied into memory:", len(memory_copy.getvalue()), "bytes")
print("no file was created:", not Path("nonexistent").exists())

# ---- StringIO is the text equivalent (used in 8.2 and 8.3) ----
text_buffer = io.StringIO()
text_buffer.write("in-memory text\n")
print("\nStringIO:", repr(text_buffer.getvalue()))

### Magic bytes: what *is* this file?

A file extension is a **hint**, not a fact. Anyone can rename `virus.exe` to `photo.jpg`.

Most binary formats start with a distinctive **signature** — "magic bytes" — in their first
few bytes. Reading those tells you what the file actually is.

| Format | Magic bytes | As text |
|---|---|---|
| JPEG | `FF D8 FF` | — |
| PNG | `89 50 4E 47 0D 0A 1A 0A` | `.PNG....` |
| GIF | `47 49 46 38` | `GIF8` |
| PDF | `25 50 44 46` | `%PDF` |
| ZIP / docx / xlsx / jar | `50 4B 03 04` | `PK..` |
| Gzip | `1F 8B` | — |
| ELF (Linux binary) | `7F 45 4C 46` | `.ELF` |
| Windows EXE/DLL | `4D 5A` | `MZ` |

**Real-world use case:** validating uploads. If a user uploads "avatar.jpg", checking the
magic bytes tells you whether it is really a JPEG before you hand it to an image library.

In [ ]:
from pathlib import Path

SIGNATURES = {
    b"\xff\xd8\xff": "JPEG image",
    b"\x89PNG\r\n\x1a\n": "PNG image",
    b"GIF8": "GIF image",
    b"%PDF": "PDF document",
    b"PK\x03\x04": "ZIP archive (or docx/xlsx/jar)",
    b"\x1f\x8b": "Gzip archive",
    b"\x7fELF": "ELF executable",
    b"MZ": "Windows executable",
    b"{": "probably JSON",
}


def identify(path: Path) -> str:
    """Identify a file from its first bytes, ignoring the extension."""
    with open(path, "rb") as handle:
        header = handle.read(16)

    for signature, description in SIGNATURES.items():
        if header.startswith(signature):
            return description

    # No signature matched - is it plausibly text?
    try:
        header.decode("utf-8")
        return "text (no binary signature)"
    except UnicodeDecodeError:
        return f"unknown binary (starts {header[:4].hex(' ')})"


folder = Path("File2Save")
print(f"{'file':<26} {'extension':<11} actual content")
print("-" * 62)
for path in sorted(folder.iterdir()):
    if path.is_file():
        print(f"{path.name:<26} {path.suffix or '(none)':<11} {identify(path)}")


# ---- The point: a lying extension is caught ----
liar = Path("File2Save/definitely_an_image.jpg")
liar.write_text("I am plain text pretending to be a JPEG\n", encoding="utf-8")

print(f"\n{liar.name}")
print("  extension says :", liar.suffix)
print("  contents say   :", identify(liar))
print("  ^ this is why upload validation reads the bytes, not the name")

liar.unlink()

### `struct`: fixed-width binary records

Binary file formats and network protocols pack numbers into a fixed number of bytes.
`struct` converts between those raw bytes and Python values.

### Syntax breakdown

```
struct.pack("<IHf", 1, 2, 3.0)
            |||||
            ||||+-- f = 4-byte float
            |||+--- H = 2-byte unsigned short
            ||+---- I = 4-byte unsigned int
            |+----- (repeat as needed)
            +------ < = little-endian, > = big-endian, ! = network order
```

| Code | Type | Bytes |
|---|---|---|
| `b` / `B` | signed / unsigned char | 1 |
| `h` / `H` | short | 2 |
| `i` / `I` | int | 4 |
| `q` / `Q` | long long | 8 |
| `f` / `d` | float / double | 4 / 8 |
| `s` | bytes | as specified |

> **Always specify the byte order** (`<`, `>` or `!`). The default is native, which differs
> between machines — the classic cause of "the file works on my laptop but not on the server".

**Real-world use case:** reading a WAV or BMP header, parsing a binary log format, or
implementing a network protocol (**11 Socket Programming**).

In [ ]:
import struct
from pathlib import Path

# ---- Pack values into bytes ----
record = struct.pack("<IHf", 1001, 42, 19.99)      # < = little-endian
print("packed  :", record, f"({len(record)} bytes)")
print("expected:", struct.calcsize("<IHf"), "bytes")

# ---- Unpack them back ----
order_id, quantity, price = struct.unpack("<IHf", record)
print(f"unpacked: id={order_id} qty={quantity} price={price:.2f}")


# ---- ⚠️ Byte order matters ----
little = struct.pack("<I", 1)
big = struct.pack(">I", 1)
print("\nthe integer 1 as little-endian:", little.hex(" "))
print("the integer 1 as big-endian   :", big.hex(" "))
print("misreading little as big      :", struct.unpack(">I", little)[0])
print("  ^ 1 became 16,777,216 - a silent, catastrophic misread")


# ---- A real fixed-width record file ----
RECORD = "<I16sf"                       # id, 16-byte name, price
orders = [(1001, b"Keyboard", 1299.50),
          (1002, b"Monitor", 12499.00),
          (1003, b"USB-C cable", 349.99)]

path = Path("File2Save/orders.bin")
with open(path, "wb") as handle:
    for oid, name, price in orders:
        handle.write(struct.pack(RECORD, oid, name.ljust(16, b"\x00"), price))

size = struct.calcsize(RECORD)
print(f"\nwrote {path.stat().st_size} bytes = {path.stat().st_size // size} records of {size}")

# Reading it back - one fixed-size record at a time
print("\nrecords:")
with open(path, "rb") as handle:
    while chunk := handle.read(size):
        oid, raw_name, price = struct.unpack(RECORD, chunk)
        name = raw_name.rstrip(b"\x00").decode("utf-8")
        print(f"  {oid}  {name:<14} {price:>10,.2f}")

print("\nfixed-width means record N starts at N * size - so you can SEEK")
with open(path, "rb") as handle:
    handle.seek(1 * size)               # jump straight to the second record
    oid, raw_name, price = struct.unpack(RECORD, handle.read(size))
    print(f"  record[1] without reading record[0]: {oid} "
          f"{raw_name.rstrip(chr(0).encode()).decode()}")

path.unlink()

---

## Common Mistakes & Pitfalls

1. 🔴 **Opening a binary file in text mode.** It either raises `UnicodeDecodeError` or, worse, silently corrupts the data through newline translation.
2. 🔴 **Passing `encoding=` with a binary mode.** `ValueError: binary mode doesn't take an encoding argument`.
3. **`file.read()` on a huge file.** Read in chunks.
4. **Writing `str` to a binary file.** `TypeError: a bytes-like object is required`. Encode first (**2.1**).
5. **Trusting a file extension.** Check the magic bytes when it matters.
6. **Omitting the byte-order prefix in `struct`.** Native order differs between machines, so the file stops being portable.
7. **Using MD5 or SHA-1 for security.** Both are broken against deliberate collisions. Use SHA-256.
8. **Using any general-purpose hash for passwords.** Use `hashlib.scrypt`, `argon2` or `bcrypt`.
9. **Indexing `bytes` and expecting `bytes`.** `data[0]` is an `int`; `data[0:1]` is `bytes` (**2.1**).

## Best Practices

- Use `"rb"`/`"wb"` for anything that is not text you control.
- Read large files in chunks — 64 KB is a sensible default.
- Write a `chunks()` generator once and reuse it for hashing, copying and scanning.
- Use `hashlib.file_digest()` (3.11+) rather than hand-rolling the loop.
- Use SHA-256 for integrity; BLAKE2 when you need speed.
- Use `shutil.copyfileobj` rather than `read()` then `write()`.
- Always specify endianness in `struct` format strings — prefer `!` for network data.
- Use `io.BytesIO` to test binary code without touching the disk.
- Validate uploads by their magic bytes, never by their filename.

## Practice Exercises

Try these before moving on.

1. Open a JPEG in text mode with `errors="replace"` and count how many characters were corrupted.
2. Write `chunks()` and use it to count how many times a given byte appears in a large file.
3. Compute the SHA-256 of a file two ways — your own loop and `hashlib.file_digest` — and confirm they match.
4. Find every duplicate file in a directory tree by content hash.
5. Flip one bit in a copy of a file and show the hash changes completely.
6. Write `identify()` for five more formats and test it on renamed files.
7. Use `struct` to write 1,000 fixed-width records, then seek directly to record 500 without reading the others.
8. Pack an integer as big-endian and unpack it as little-endian. Explain the result.
9. Build a small file entirely in `BytesIO` and only then write it to disk.